In [0]:
from pyspark.sql.functions import col, count,desc
import time
from datetime import datetime

In [0]:
def clean_data(df):
    known_bots = [
    "AutoModerator", "RemindMeBot", "QualityVote", "Bot_Metric", 
    "haikusbot", "WikiTextBot", "vredditDownloader", "image_linker_bot",
    "TrollaBot", "bot", "Bot" 
]

    # lọc bots
    df_clean = df.filter(~col("author").isin(known_bots)) \
                .filter(~col("author").rlike("(?i)bot")) \
                .filter(col("author").isNotNull()) \
                .filter(col("subreddit").isNotNull())

    return df_clean

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, desc

start = datetime.now()

path = "wasbs://reddit-data@chungminhlamkhaiphadl.blob.core.windows.net/pushshift_1/*submissions_cleaned.parquet"
df_all = spark.read.parquet(path)

df_clean = clean_data(df_all)

windowSpec = Window.partitionBy("subreddit").orderBy(desc("score"))

df_ranked = df_clean.withColumn("rank", row_number().over(windowSpec))

df_top_1000 = df_ranked.filter(col("rank") <= 1000).drop("rank")

df_final = df_top_1000.select("subreddit", "title", "score", "subreddit_subscribers", "domain")

output_path = "wasbs://data@chungminhlamkhaiphadl.blob.core.windows.net/embedding_data"

df_final.write.mode("overwrite").partitionBy("subreddit").parquet(output_path)

end = datetime.now()
print("Start:", start)
print("End:", end)
print("Duration:", end - start)

Start: 2026-04-02 07:31:05.084078
End: 2026-04-02 07:42:42.828322
Duration: 0:11:37.744244


In [0]:
def get_dir_size(path):
    total_size = 0
    files = dbutils.fs.ls(path)
    for f in files:
        if f.isDir():
            total_size += get_dir_size(f.path)
        else:
            total_size += f.size
    return total_size

size_bytes = get_dir_size("wasbs://data@chungminhlamkhaiphadl.blob.core.windows.net/embedding_data")
print(f"Tổng kích thước: {size_bytes / (1024*1024):.2f} MB")

Tổng kích thước: 153.29 MB
